# 04 · Advanced RAG I: Pre-retrieval
**Concepts:** (A) chunking strategies, (B) multi-query rewriting, (C) metadata filtering extracted from the question.
**TAC scenario:** clarify the question, try several phrasings, narrow to the right product line.

In [ ]:
from rag_common import MODEL, EMBED_MODEL, get_chat_model, get_vectorstore

model = get_chat_model()
vectorstore = get_vectorstore()          # opens the Chroma store built in Notebook 01
print(f"Chat: {MODEL} | Embeddings: {EMBED_MODEL} | chunks in store: {vectorstore._collection.count()}")

In [ ]:
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
from rag_common import load_markdown_dir, get_embeddings, golden_set, doc_ids, show, hit, chroma_filter

embeddings = get_embeddings()
kb_docs = load_markdown_dir("kb")
kb_questions = [g for g in golden_set() if g["type"] in ("single_hop", "error_code", "version_specific")
                and all(e.startswith("KB-") for e in g["expected_doc_ids"])]
print(len(kb_docs), "KB articles |", len(kb_questions), "KB golden questions")

## A. Chunking strategies: same KB, three ways

In [ ]:
def fixed(size, overlap):
    return RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap).split_documents(kb_docs)

def by_headers():
    hs = MarkdownHeaderTextSplitter(headers_to_split_on=[("#", "title"), ("##", "section")])
    out = []
    for d in kb_docs:
        for part in hs.split_text(d.page_content):
            part.page_content = f"{part.metadata.get('title', '')}\n{part.metadata.get('section', '')}: {part.page_content}"
            part.metadata = {**d.metadata, **part.metadata}
            out.append(part)
    return out

strategies = {"small 200/0": fixed(200, 0), "large 1500/150": fixed(1500, 150), "header-aware": by_headers()}
stores = {}
for name, chunks in strategies.items():
    stores[name] = Chroma.from_documents(chunks, embeddings, collection_name=f"chunk_{len(stores)}")   # in-memory
    print(f"{name:15s} {len(chunks):4d} chunks, avg {sum(len(c.page_content) for c in chunks)//len(chunks)} chars")

In [ ]:
K = 3
print(f"hit@{K} on {len(kb_questions)} KB questions")
for name, vs in stores.items():
    hits = sum(hit(doc_ids(vs.similarity_search(g["question"], k=K)), g["expected_doc_ids"]) for g in kb_questions)
    print(f"  {name:15s} {'█' * hits}{'░' * (len(kb_questions) - hits)} {hits}/{len(kb_questions)}")

> Results depend on the corpus. Structure-aware chunks keep each section (Symptoms, Root cause…) whole and labelled.

## B. Multi-query: several phrasings, merged results

In [ ]:
from pydantic import BaseModel, Field

class Rewrites(BaseModel):
    queries: list[str] = Field(description="3 alternative search queries using different technical wording")

rewriter = model.with_structured_output(Rewrites)
question = "The overlay keeps bouncing when branches fail over to 4G"
rewrites = rewriter.invoke(f"Rewrite this support question into 3 different search queries:\n{question}").queries
for r in rewrites:
    print("•", r)

In [ ]:
base = vectorstore.as_retriever(search_kwargs={"k": 4})
single = doc_ids(base.invoke(question))
merged = list(dict.fromkeys(i for q in [question, *rewrites] for i in doc_ids(base.invoke(q))))
print("single query :", single)
print("multi-query  :", merged, f"(+{len(set(merged) - set(single))} new documents)")

In [ ]:
# The built-in one-liner does the same (rewrite → retrieve each → unique union)
from langchain_classic.retrievers import MultiQueryRetriever
mq = MultiQueryRetriever.from_llm(retriever=base, llm=model)
doc_ids(mq.invoke(question))

## C. Metadata filtering: extract product and version from the question

In [ ]:
from typing import Literal, Optional

class SearchFilters(BaseModel):
    product: Optional[Literal["Aurora AR-900", "Nimbus NF-300", "Orion OW-20", "Atlas SD-WAN", "Helix HX-5000"]] = None
    version: Optional[str] = Field(None, description="Software version mentioned, e.g. '9.1.4'")

extractor = model.with_structured_output(SearchFilters)

def filtered_search(q: str, k: int = 4):
    f = extractor.invoke(f"Extract the product and version from this question, if present:\n{q}")
    flt = chroma_filter(product=f.product, version=f.version)
    docs = vectorstore.similarity_search(q, k=k, filter=flt) if flt else []
    return f, flt, (docs or vectorstore.similarity_search(q, k=k))        # fall back to unfiltered if empty

In [ ]:
for g in [x for x in golden_set() if x["type"] == "version_specific"]:
    print(f"── {g['question']}\n   expected {g['expected_doc_ids']}")
    print("   unfiltered:", doc_ids(vectorstore.similarity_search(g["question"], k=4)))
    f, flt, docs = filtered_search(g["question"])
    print(f"   filtered  : {doc_ids(docs)}   ← {f.product} {f.version}\n")

---
**🔧 TAC takeaway:** Seniors search smart: clarify the question, try several phrasings, and narrow to the right product line before they look at a single result.